**This notebook is ONLY for data, no CNN yet.**

✅ Tasks in 01_data_check.ipynb

Mount Google Drive

Read dataset folder structure

Count images per class

Detect missing / corrupted images

Show random samples

Verify image size & channels

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

DATASET_PATH = "/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/Datasets"


In [7]:
os.listdir(DATASET_PATH)



['Brinjal', 'Castor', 'Papaya', 'Cumin', 'Guava']

In [8]:
import os

for crop in os.listdir(DATASET_PATH):
    print(f"\nCrop: {crop}")
    crop_path = os.path.join(DATASET_PATH, crop)

    for status in os.listdir(crop_path):
        status_path = os.path.join(crop_path, status)

        # Skip files accidentally present
        if not os.path.isdir(status_path):
            print("  ❌ File found (should not be here):", status)
            continue

        print("  ", status)

        # Healthy → images directly
        if status.lower() == "healthy":
            imgs = os.listdir(status_path)
            print("     Images:", len(imgs))

        # Unhealthy → disease-wise folders
        elif status.lower() == "unhealthy":
            for disease in os.listdir(status_path):
                disease_path = os.path.join(status_path, disease)

                if not os.path.isdir(disease_path):
                    continue

                img_count = len(os.listdir(disease_path))
                print("     Disease:", disease, "| Images:", img_count)



Crop: Brinjal
   Healthy
     Images: 54
   Unhealthy
     Disease: Bacrerial_Wilt | Images: 17
     Disease: Mosaic | Images: 14
     Disease: Cercospora_Leaf_spot | Images: 47
     Disease: Phomopsis_Leaf_Blight | Images: 20

Crop: Castor
   Unhealthy
     Disease: leaf_curv_virus | Images: 14
     Disease: Bacterial_Leaf_Blight | Images: 20
     Disease: Alternaria_Leaf_Blight | Images: 21
     Disease: Cercospora_Leaf_Spot | Images: 22
   Healthy
     Images: 35

Crop: Papaya
   Unhealthy
     Disease: Leaf_Spot | Images: 13
     Disease: Powdery_Mildew | Images: 15
     Disease: Ring_Spot_Virus | Images: 15
   Healthy
     Images: 34

Crop: Cumin
   Healthy
     Images: 24
   Unhealthy
     Disease: Alternaria_Blight | Images: 17
     Disease: Wilt | Images: 11

Crop: Guava
   Healthy
     Images: 45
   Unhealthy
     Disease: Bacterial_Blight | Images: 17
     Disease: Anthracnose | Images: 32
     Disease: Wilt | Images: 24
     Disease: Red_Rust | Images: 16


In [9]:
SOURCE = "/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/Datasets"
DEST   = "/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/split_dataset"

In [10]:
split_ratio = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}

In [11]:
import shutil
import random

for crop in os.listdir(SOURCE):
    crop_path = os.path.join(SOURCE, crop)

    for status in os.listdir(crop_path):
        status_path = os.path.join(crop_path, status)

        # Healthy images directly
        if status.lower() == "healthy":
            images = os.listdir(status_path)
            random.shuffle(images)

            total = len(images)
            train_end = int(total * 0.7)
            val_end   = int(total * 0.85)

            splits = {
                "train": images[:train_end],
                "val": images[train_end:val_end],
                "test": images[val_end:]
            }

            for split in splits:
                target_dir = os.path.join(DEST, split, crop, "Healthy")
                os.makedirs(target_dir, exist_ok=True)

                for img in splits[split]:
                    shutil.copy(
                        os.path.join(status_path, img),
                        os.path.join(target_dir, img)
                    )

        # Unhealthy → disease folders
        elif status.lower() == "unhealthy":
            for disease in os.listdir(status_path):
                disease_path = os.path.join(status_path, disease)

                images = os.listdir(disease_path)
                random.shuffle(images)

                total = len(images)
                train_end = int(total * 0.7)
                val_end   = int(total * 0.85)

                splits = {
                    "train": images[:train_end],
                    "val": images[train_end:val_end],
                    "test": images[val_end:]
                }

                for split in splits:
                    target_dir = os.path.join(DEST, split, crop, disease)
                    os.makedirs(target_dir, exist_ok=True)

                    for img in splits[split]:
                        shutil.copy(
                            os.path.join(disease_path, img),
                            os.path.join(target_dir, img)
                        )

# **Verify Split**

In [12]:
for split in ["train", "val", "test"]:
    print(f"\n--- {split.upper()} ---")
    split_path = os.path.join(DEST, split)

    for crop in os.listdir(split_path):
        print(f"\nCrop: {crop}")
        crop_path = os.path.join(split_path, crop)

        for disease in os.listdir(crop_path):
            disease_path = os.path.join(crop_path, disease)
            count = len(os.listdir(disease_path))
            print(f"   {disease} → {count} images")


--- TRAIN ---

Crop: Brinjal
   Healthy → 54 images
   Bacrerial_Wilt → 16 images
   Mosaic → 14 images
   Cercospora_Leaf_spot → 47 images
   Phomopsis_Leaf_Blight → 20 images

Crop: Castor
   Healthy → 35 images
   leaf_curv_virus → 14 images
   Bacterial_Leaf_Blight → 20 images
   Alternaria_Leaf_Blight → 20 images
   Cercospora_Leaf_Spot → 22 images

Crop: Papaya
   Leaf_Spot → 13 images
   Powdery_Mildew → 14 images
   Ring_Spot_Virus → 15 images
   Healthy → 33 images

Crop: Cumin
   Healthy → 24 images
   Alternaria_Blight → 17 images
   Wilt → 11 images

Crop: Guava
   Healthy → 45 images
   Bacterial_Blight → 16 images
   Anthracnose → 32 images
   Wilt → 24 images
   Red_Rust → 14 images

--- VAL ---

Crop: Brinjal
   Healthy → 26 images
   Bacrerial_Wilt → 9 images
   Mosaic → 7 images
   Cercospora_Leaf_spot → 24 images
   Phomopsis_Leaf_Blight → 8 images

Crop: Castor
   Healthy → 17 images
   leaf_curv_virus → 7 images
   Bacterial_Leaf_Blight → 10 images
   Alternaria_L